# 프로젝트 요약 — 무엇을 만들었고, 어떻게 풀어갔나

공공기관 AI·IT 사업 제안요청서(RFP)에는 요구사항 조항이 수십~수백 개 있다. 제안서를 쓰는 실무자는
그걸 다 같은 무게로 읽지 않는다. 대부분은 그냥 받아들이고, 일부는 견적에 넣고, 몇 개는 계약 전에
따져본다. 그 분류를 모델이 미리 해주면 실무자는 마지막 것부터 읽으면 된다.

- `통상수용` — 표준 기술·일반 관행이라 별도 원가 없이 받아들임
- `견적반영` — 인력·장비·라이선스 등 추가 원가가 계산됨
- `계약·질의검토` — 특정 벤더 지정, 발주기관이 정한 성능 수치, 범위가 열려 있음 등 계약 전에 확인이 필요

이 노트북은 그 과정을 한 자리에서 다시 계산한다. 숫자는 전부 `reports/current/`와 `data/labels/`에서
읽고 여기서 새로 학습하는 것은 없다. 어느 셀도 `RFP_DATASET_VERSION`에 기대지 않는다.

## 흐름

| 단계 | 한 줄 | 자세한 노트북 |
|---|---|---|
| 1. 데이터 | 나라장터 공고 13건의 hwp·xlsx에서 요구사항 표를 직접 파싱, 1,445건 | [01](01_dataset_pipeline.ipynb) [03](03_requirements_eda.ipynb) |
| 2. 라벨 | 사람 대신 Claude. 판정 규칙 + 문서별 예시(앵커)를 같이 주고 배치로 라벨링, 조건을 바꿔 재실행해 흔들림 측정 | [02](02_labeling_experiment.ipynb) [04](04_anchor_pool_analysis.ipynb) [06](06_label_eda.ipynb) [17](17_rerun_agreement.ipynb) |
| 3. 모델 | 13문서 LODO를 고정하고 TF-IDF → 트리 → 임베딩 → 마스킹 → 파인튜닝 → 앙상블 → sLLM 순서로 올림 | [07](07_baseline_comparison.ipynb) [10](10_explainable_classical_search.ipynb) [08](08_embedding_comparison.ipynb) [14](14_text_masking.ipynb) [15](15_finetuning.ipynb) [19](19_training_recipes.ipynb) [12](12_candidate_ensemble.ipynb) |
| 4. 오답 분석 | 앙상블도 못 줄인 오답을 들여다보고, 라벨 규칙을 고쳐(v7) 다시 잼 | [13](13_label_boundary.ipynb) [20](20_boundary_cases.ipynb) [21](21_cluster_diagnostics.ipynb) [18](18_decision_structure.ipynb) |

결정 기록은 `docs/history/decisions-*.md`, 남은 일은 `docs/NEXT.md`.


In [ ]:
"""1. 데이터 — 문서 13개, 조항 1,445건. 라벨셋은 v5(주)와 v7(규칙을 좁힌 것) 둘을 나란히 본다."""
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts.labeling.label_dataset import load_label_dataset

LABELS = ("통상수용", "견적반영", "계약·질의검토")
rows = {v: load_label_dataset(version=v)[0] for v in ("v5", "v7")}
docs = sorted({r["document_id"] for r in rows["v5"]})
print(f"문서 {len(docs)}개, 조항 {len(rows['v5'])}건")

dist = pd.DataFrame({v: pd.Series([r["primary_action"] for r in rs]).value_counts() for v, rs in rows.items()}).loc[list(LABELS)]
dist_pct = (dist / dist.sum() * 100).round(1).astype(str) + "%"
print("\n라벨 분포"); print(pd.concat({"건수": dist, "비율": dist_pct}, axis=1).to_string())

per_doc = pd.crosstab(pd.Series([r["document_id"] for r in rows["v5"]], name="문서"),
                      pd.Series([r["primary_action"] for r in rows["v5"]], name="v5 라벨"))[list(LABELS)]
per_doc["합"] = per_doc.sum(axis=1)
print("\n문서별 (v5)"); print(per_doc.sort_values("합", ascending=False).to_string())
print(f"\n→ 문서마다 크기가 {per_doc['합'].min()}~{per_doc['합'].max()}건, 라벨 비율도 제각각이다. 무작위로 섞어 나누면 같은 문서의 조항이 학습·시험에 같이 들어가 점수가 부풀므로, 문서 하나를 통째로 빼서 시험한다(13문서 LODO).")


In [ ]:
"""3. 모델 — 해본 순서대로. 무엇을, 왜, 결과는.

기준은 fold 평균 macro F1. 이전 마스킹 변동에서 도출한 0.016을 후속 실험 선별 기준으로 사용했다. 모든 실험의 유의성 기준은 아니다.
안 그러면 seed 하나 바꿔 오른 0.01을 성과로 적게 된다.
"""
import matplotlib.pyplot as plt
from matplotlib import font_manager

installed = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams["font.family"] = next((f for f in ("Malgun Gothic", "NanumGothic", "AppleGothic", "Noto Sans CJK KR") if f in installed), plt.rcParams["font.family"])
plt.rcParams["axes.unicode_minus"] = False

def load(name):
    return json.loads((ROOT / "reports/current/v5" / name).read_text(encoding="utf-8"))

cand = load("model_candidates.json")["candidates"]
ft = load("finetune_results.json")
mask = load("text_masking_ablation.json")["specs"]["word 1-2 + char 3-4gram + balanced"]
clus = load("cluster_universality.json")

def fm(c): return cand[c]["summary"]["macro_f1"]["fold_mean"]
wc = fm("word_char_logistic")
best_mask = max((v["macro_f1"], k) for k, v in mask.items() if k != "none")
best_clus = max((v["macro_f1"], k) for k, v in clus.items() if k != "baseline")
large = [ft["singles"][k]["fold_mean_macro_f1"] for k in ("ftL42", "ftL7", "ftL13")]
base = [ft["singles"][k]["fold_mean_macro_f1"] for k in ("ftB42", "ftB7", "ftB13")]
llm = ft["singles"].get("llm42", {}).get("fold_mean_macro_f1")
llm_ens = ft["ensembles"].get("wc+ftL42+llm42", {}).get("fold_mean_macro_f1")

steps = [
    ("문자 3~4gram TF-IDF + 로지스틱", "한국어는 띄어쓰기가 불규칙해 형태소 분석 없이 글자 n-gram부터", fm("char_logistic"), "첫 기준선"),
    ("단어 1~2gram + 문자 n-gram", "'협의하여', '준수하여야' 같은 단어 단위 신호를 더함", wc, "기준선 채택"),
    ("XGBoost (트리)", "선형이 못 잡는 조합 규칙이 있을까", 0.5946, "더 낮음. 용량 올리면 0.561로 더 떨어짐"),
    ("E5 문장 임베딩 + TF-IDF", "글자가 아니라 뜻으로 보면 나아질까", fm("tfidf_e5_hybrid"), "더 낮음. 동결 임베딩은 이 경계를 못 가름"),
    (f"양식 신호 마스킹 ({best_mask[1]})", "주어·어미 같은 문서 양식을 외우는 건 아닌지 지워서 확인", best_mask[0], "차이 없음(잡음 범위). 양식이 아니라 내용을 보고 있었음"),
    (f"임베딩 군집 특징 ({best_clus[1]})", "이 조항이 어느 군집에 속하는지를 특징으로 주면?", best_clus[0], "더 낮음. 군집은 라벨과 맞지 않음"),
    ("klue/roberta-base 파인튜닝 (seed 3개)", "문맥을 읽는 인코더면 TF-IDF를 넘을까", sum(base) / 3, f"seed 범위 {min(base):.3f}~{max(base):.3f}, 기준선 대비 {sum(base)/3-wc:+.3f}. 같음"),
    ("klue/roberta-large 파인튜닝 (seed 3개)", "크기를 키우면", sum(large) / 3, f"seed 범위 {min(large):.3f}~{max(large):.3f}, 기준선 대비 {sum(large)/3-wc:+.3f}. 하한 근처"),
    ("앙상블 TF-IDF + base + large 다수결", "셋이 틀리는 자리가 다르면 투표로 줄일 수 있다", ft["ensembles"]["wc+ftB7+ftL42"]["fold_mean_macro_f1"], f"기준선 대비 {ft['ensembles']['wc+ftB7+ftL42']['fold_mean_macro_f1']-wc:+.3f}. 유일하게 하한을 넘는 개선. 지정 기준선; 최신 중첩 선택은 Qwen 조합"),
    ("seed 7개 전부 투표", "멤버를 늘리면 더 오를까", 0.677, "+0.006, 하한 아래. 여기서 멤버 늘리기는 끝"),
    ("sLLM Qwen2.5-7B LoRA", "라벨 지시와 사전학습 지식을 가진 디코더가 다른 오답을 보완할까", llm, "실행 중" if llm is None else "v5 단독 0.644; 분석은 노트북 23"),
    ("앙상블 + sLLM", "sLLM이 틀리는 자리가 인코더와 다르면 투표 이득", llm_ens, "실행 중" if llm_ens is None else "v5 단독 0.644; 분석은 노트북 23"),
]
table = pd.DataFrame(steps, columns=["시도", "왜 해봤나", "fold 평균 macro F1", "결과"])
table.index = range(1, len(table) + 1)
with pd.option_context("display.max_colwidth", 60, "display.width", 200):
    print(table.assign(**{"fold 평균 macro F1": table["fold 평균 macro F1"].map(lambda x: "—" if pd.isna(x) else f"{x:.3f}")}).to_string())

plot = table.dropna(subset=["fold 평균 macro F1"])
fig, ax = plt.subplots(figsize=(8, 0.42 * len(plot) + 1))
colors = ["#2B5DA8" if v >= wc + 0.016 else "#B9C4D6" if v < wc - 0.016 else "#7F93B3" for v in plot["fold 평균 macro F1"]]
ax.barh(plot["시도"], plot["fold 평균 macro F1"], color=colors)
ax.axvline(wc, color="#333", lw=1, ls="--"); ax.text(wc, -0.7, f"기준선 {wc:.3f}", fontsize=8, ha="center")
ax.axvspan(wc - 0.016, wc + 0.016, color="#999", alpha=0.15)
ax.set_xlim(0.5, 0.72); ax.invert_yaxis(); ax.set_xlabel("fold 평균 macro F1 (회색 띠 = 기준선 ± 측정 하한)")
for y, v in enumerate(plot["fold 평균 macro F1"]):
    ax.text(v + 0.002, y, f"{v:.3f}", va="center", fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
"""4. 오답 분석 — 앙상블이 오답을 줄이는 동안 안 줄어든 자리가 있었다.

오답을 라벨 쌍으로 갈라 보니 견적↔계약 경계에 몰려 있고, 앙상블도 거기는 거의 못 줄였다.
그래서 그 조항들과 뜻이 비슷한 조항이 **다른 문서에서는** 어떤 라벨을 받았는지 봤다.
"""
S, E = ft["singles"], ft["ensembles"]
err = pd.DataFrame({
    "오답": {"TF-IDF": S["word+char TF-IDF"]["errors"], "roberta-large": S["ftL42"]["errors"], "앙상블": E["wc+ftB7+ftL42"]["errors"]},
    "그중 견적↔계약 경계": {"TF-IDF": S["word+char TF-IDF"]["boundary_errors"], "roberta-large": S["ftL42"]["boundary_errors"], "앙상블": E["wc+ftB7+ftL42"]["boundary_errors"]},
})
err["경계 비중"] = (err["그중 견적↔계약 경계"] / err["오답"] * 100).round(0).astype(int).astype(str) + "%"
print("오답 구조 (v5, 평가 1,345건)"); print(err.to_string())

cons = load("cluster_diagnostics.json")["cross_document_label_consistency"]
cons_df = pd.DataFrame({g: {"건수": cons[g]["n"], "다른 문서의 비슷한 조항 5건과 라벨 일치": f"{cons[g]['rate']*100:.0f}%"}
                        for g in ("모델이 맞힌 건", "경계 밖 오답", "경계 혼동")}).T
print("\n비슷한 조항이 다른 RFP에서 같은 라벨을 받는 비율 (E5 이웃, 노트북 21)"); print(cons_df.to_string())
print("\n→ 맞힌 조항은 다른 문서에서도 같은 라벨(79%)인데, 경계 오답은 4건 중 1건만 같다. 같은 문장이 다른 곳에서는 반대로 라벨돼 있으니")
print("   유사 문장의 라벨 불일치가 확인됐다. 표본·규칙·문서 조건 중 원인을 이 수치만으로 분리할 수는 없다.")

# Claude가 127건을 검토하고 실무자가 27건을 판정했다. 그 결과로 v7 규칙을 만들었다.
v7 = json.loads((ROOT / "reports/current/v7/finetune_results.json").read_text(encoding="utf-8"))
cmp = pd.DataFrame({
    "v5": {"TF-IDF": S["word+char TF-IDF"]["fold_mean_macro_f1"], "roberta-large": S["ftL42"]["fold_mean_macro_f1"], "앙상블": E["wc+ftB7+ftL42"]["fold_mean_macro_f1"]},
    "v7 (규칙을 좁힌 라벨)": {"TF-IDF": v7["singles"]["word+char TF-IDF"]["fold_mean_macro_f1"], "roberta-large": v7["singles"]["ftL42"]["fold_mean_macro_f1"], "앙상블": v7["ensembles"]["wc+ftB7+ftL42"]["fold_mean_macro_f1"]},
})
cmp["차이"] = cmp.iloc[:, 1] - cmp.iloc[:, 0]
print("\n라벨 규칙을 실무 판단에 맞게 좁히면 (v7)"); print(cmp.round(3).to_string())
print("\n→ 실무자 판정 27건과의 일치는 11/26 → 18/26으로 올랐는데 점수는 내려갔다. 계약 라벨이 25%→20%로 얇아지자 그 클래스 F1이 떨어져서다.")
print("   대신 TF-IDF는 0.035를 잃고 인코더는 0.016만 잃는다. 라벨이 '글자'가 아니라 '뜻'으로 갈릴수록 문맥을 읽는 모델이 유리하다.")
print("   Qwen LoRA 완료: v7 단독 0.665. 성능 비교와 입력 누락 진단은 노트북 23에서 재현한다.")


## 정리

v5 주 구성은 기존 앙상블 0.671입니다. Qwen을 포함한 조합은 0.683으로 약 0.011 높았지만 후속 실험 선별 기준 0.016에 못 미쳤습니다.
v7 Qwen 단독은 0.665로 large보다 높았지만 경계 혼동은 늘었습니다.

Qwen 결과 정독에서는 512토큰 밖의 성능·공급 조건이 빠진 사례도 확인했습니다.
짧은 조항의 오답과 실무 판단 차이도 있어, 원인을 문서 밖 지식 하나로 단정하지 않습니다.

- 자세한 학습 방식·재계산·입력 진단: [23 Qwen LoRA](23_qwen_lora_analysis.ipynb)
- 포트폴리오 콘텐츠: [상세 Markdown](../docs/portfolio/rfp_portfolio_detailed.md)
- 후속 작업과 채택 판단: [NEXT](../docs/NEXT.md)

데이터 추가 전후에는 평가 모집단도 달라졌고, v5·v7은 정답 라벨이 다릅니다.
작은 점수 차이를 확정 성과로 만들기보다 조건·오답·판단 근거를 함께 기록했습니다.